<a href="https://colab.research.google.com/github/adalves-ufabc/2026.Q3-PLN/blob/main/2026_Q3_PLN_AULA_11_Notebook_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Processamento de Linguagem Natural [2026-Q3]**
Prof. Alexandre Donizeti Alves

## **Introdução à API Gemini do Google**
---



A API Gemini permite acessar os modelos generativos mais recentes do Google.

**Instalar o SDK da API Gemini**

O SDK do Python para a API Gemini está localizado `google-genai` . Instale a dependência usando `pip`:

In [ ]:
!pip install google-genai

In [ ]:
#@title Versão da API Gemini

from importlib.metadata import version

print(version("google-genai"))

2.12.1


**Configurar sua chave de API**

Para usar a API Gemini, você precisa de uma chave de API. Se você ainda não tiver uma, criar uma chave no Google AI Studio.

No Colab, adicione a chave ao gerenciador de chaves secretas em "Secrets" 🔑 no painel esquerdo. Nomeie como `GOOGLE_API_KEY`.

In [ ]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
#@title Definindo a chave da API [$$$]

from getpass import getpass

GEMINI_API_KEY = getpass()

··········


In [ ]:
#@title Total de Modelos

from google import genai

cliente = genai.Client(api_key=GEMINI_API_KEY)

modelos = list(cliente.models.list())

print(f"Total geral de modelos: {len(modelos)}")

Total geral de modelos: 63


In [ ]:
#@title Lista de Modelos

from google import genai

cliente = genai.Client(api_key=GEMINI_API_KEY)

total_modelos = 0

print("Modelos que suportam generateContent:\n")

for modelo in cliente.models.list():
    if "generateContent" in (modelo.supported_actions or []):
        print(modelo.name)
        total_modelos += 1

print(f"\nTotal de modelos: {total_modelos}")

Modelos que suportam generateContent:

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-prev

In [ ]:
print("Lista de modelos que suportam embedContent:\n")
for m in cliente.models.list():
    for action in m.supported_actions:
        if action == "embedContent":
            print(m.name)

Lista de modelos que suportam embedContent:

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [ ]:
print("Lista de modelos de imagem:\n")

for modelo in cliente.models.list():
    nome = modelo.name.lower()

    if any(termo in nome for termo in ["image", "imagen"]):
        print(modelo.name)

Lista de modelos de imagem:

models/gemini-2.5-flash-image
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image


In [ ]:
print("Lista de modelos que geram videos:\n")

for modelo in cliente.models.list():
    nome = modelo.name.lower()

    if "veo" in nome or "video" in nome or "omni" in nome:
        print(modelo.name)

Lista de modelos que geram videos:

models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/veo-3.1-generate-preview
models/veo-3.1-fast-generate-preview
models/veo-3.1-lite-generate-preview


In [ ]:
from google import genai

cliente = genai.Client( api_key= GEMINI_API_KEY)

model_info = cliente.models.get(model="gemini-3.5-flash-lite")
model_info

Model(
  description='Gemini 3.5 Flash Lite',
  display_name='Gemini 3.5 Flash Lite',
  input_token_limit=1048576,
  max_temperature=2.0,
  name='models/gemini-3.5-flash-lite',
  output_token_limit=65536,
  supported_actions=[
    'generateContent',
    'countTokens',
    'createCachedContent',
    'batchGenerateContent',
  ],
  temperature=1.0,
  thinking=True,
  top_k=64,
  top_p=0.95,
  tuned_model_info=TunedModelInfo(),
  version='3.5-flash-lite-07-2026'
)

### **Geração de Texto**

A **API Gemini** pode gerar saída de texto com base em entradas de texto, imagens, vídeo e áudio.

Confira um exemplo básico:

In [ ]:
from google import genai

cliente = genai.Client( api_key= GEMINI_API_KEY )

resposta = cliente.models.generate_content(
    model = "gemini-3.5-flash-lite",
    contents = "Qual a capital do Brasil?"
)

print(resposta.text)

A capital do Brasil é **Brasília**.


In [ ]:
from IPython.display import display, Markdown

display(Markdown(resposta.text))

A capital do Brasil é **Brasília**.

**Instruções do sistema e outras configurações**

Você pode orientar o comportamento dos modelos do Gemini com instruções do sistema. Para isso, transmita um `GenerateContentConfig` objeto.

In [ ]:
from google import genai
from google.genai import types

cliente = genai.Client( api_key= GEMINI_API_KEY )

resposta = resposta = cliente.models.generate_content(
    model = "gemini-3.5-flash-lite",
    config = types.GenerateContentConfig(
        system_instruction = "Você é um gato. Seu nome é Keno."),
    contents = "Olá"
)

print(resposta.text)

Miau! Olha, humano... Você veio na hora certa. Já está na hora do meu sachê, sabia? 

*Lambe a pata e esfrega na orelha* 

O que você quer? Apenas carinho, ou tem petiscos aí? 🐾


In [ ]:
from IPython.display import display, Markdown

display(Markdown(resposta.text))

Miau! Olha, humano... Você veio na hora certa. Já está na hora do meu sachê, sabia? 

*Lambe a pata e esfrega na orelha* 

O que você quer? Apenas carinho, ou tem petiscos aí? 🐾

O objeto `GenerateContentConfig` também permite substituir os parâmetros de geração padrão, como `max_output_tokens`.

In [ ]:
from google import genai
from google.genai import types

cliente = genai.Client(api_key=GEMINI_API_KEY)

resposta = cliente.models.generate_content(
    model = "gemini-3.5-flash-lite",
    contents = "Explique como a IA funciona",
    config=types.GenerateContentConfig(
        max_output_tokens = 50
    )
)

print(resposta.text)

Explicar como a Inteligência Artificial (IA) funciona pode parecer complexo, mas a ideia central é simples: **a IA tenta imitar a capacidade humana de aprender, raciocinar, tomar decisões e resolver problemas,


### **Saída Estruturada**

É possível configurar os modelos do Gemini para gerar respostas que sigam um esquema JSON fornecido. Isso garante resultados previsíveis e seguros de tipo e simplifica a extração de dados estruturados de texto não estruturado.

**Extrator de Receitas**

Este exemplo demonstra como extrair dados estruturados de texto usando tipos básicos de esquema **JSON**, como `object`, `array`, `string` e `integer`.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

# Define a estrutura de um ingrediente
class Ingrediente(BaseModel):
    nome: str = Field(description="Nome do ingrediente.")
    quantidade: str = Field(description="Quantidade do ingrediente, incluindo as unidades de medida."
    )

# Define a estrutura da receita
class Receita(BaseModel):
    nome_receita: str = Field(description="Nome da receita.")
    tempo_preparo_minutos: Optional[int] = Field(
        description="Tempo de preparo em minutos, se disponível."
    )
    ingredientes: List[Ingrediente]
    instrucoes: List[str]

# Texto de entrada para extração das informações
prompt = """
Extraia a receita a seguir e organize suas informações.

A receita é de biscoitos de chocolate.

Ingredientes:
- 2 e 1/4 xícaras de farinha de trigo
- 1 colher de chá de bicarbonato de sódio
- 1 colher de chá de sal
- 1 xícara de manteiga sem sal amolecida
- 3/4 de xícara de açúcar refinado
- 3/4 de xícara de açúcar mascavo compactado
- 1 colher de chá de extrato de baunilha
- 2 ovos grandes
- 2 xícaras de gotas de chocolate meio amargo

Modo de preparo:
1. Preaqueça o forno a 190 °C.
2. Em uma tigela pequena, misture a farinha, o bicarbonato e o sal.
3. Em uma tigela grande, bata a manteiga com o açúcar refinado e o
   açúcar mascavo até obter uma mistura leve e cremosa.
4. Acrescente a baunilha e os ovos, um de cada vez.
5. Adicione gradualmente os ingredientes secos e misture até incorporar.
6. Acrescente as gotas de chocolate e misture.
7. Distribua porções da massa em uma assadeira sem untar.
8. Asse por 9 a 11 minutos.
"""

In [ ]:
from google import genai

cliente = genai.Client( api_key=GEMINI_API_KEY )

resposta = cliente.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt,
    config={
        "response_mime_type": "application/json",
        "response_schema": Receita
    }
)

print(resposta.parsed)

nome_receita='Biscoitos de chocolate' tempo_preparo_minutos=None ingredientes=[Ingrediente(nome='farinha de trigo', quantidade='2 e 1/4 xícaras'), Ingrediente(nome='bicarbonato de sódio', quantidade='1 colher de chá'), Ingrediente(nome='sal', quantidade='1 colher de chá'), Ingrediente(nome='manteiga sem sal amolecida', quantidade='1 xícara'), Ingrediente(nome='açúcar refinado', quantidade='3/4 de xícara'), Ingrediente(nome='açúcar mascavo compactado', quantidade='3/4 de xícara'), Ingrediente(nome='extrato de baunilha', quantidade='1 colher de chá'), Ingrediente(nome='ovos grandes', quantidade='2'), Ingrediente(nome='gotas de chocolate meio amargo', quantidade='2 xícaras')] instrucoes=['Preaqueça o forno a 190 °C.', 'Em uma tigela pequena, misture a farinha, o bicarbonato e o sal.', 'Em uma tigela grande, bata a manteiga com o açúcar refinado e o açúcar mascavo até obter uma mistura leve e cremosa.', 'Acrescente a baunilha e os ovos, um de cada vez.', 'Adicione gradualmente os ingredien

In [ ]:
from IPython.display import display, Markdown

receita = resposta.parsed

display(Markdown(f"""# 🍪 {receita.nome_receita}

⏱️ **Tempo de preparo:** {receita.tempo_preparo_minutos} minutos

## 🛒 Ingredientes
{chr(10).join(f"- **{i.quantidade}** — {i.nome}" for i in receita.ingredientes)}

## 👨‍🍳 Modo de preparo
{chr(10).join(f"{n}. {etapa}" for n, etapa in enumerate(receita.instrucoes, 1))}
"""))

# 🍪 Biscoitos de chocolate

⏱️ **Tempo de preparo:** None minutos

## 🛒 Ingredientes
- **2 e 1/4 xícaras** — farinha de trigo
- **1 colher de chá** — bicarbonato de sódio
- **1 colher de chá** — sal
- **1 xícara** — manteiga sem sal amolecida
- **3/4 de xícara** — açúcar refinado
- **3/4 de xícara** — açúcar mascavo compactado
- **1 colher de chá** — extrato de baunilha
- **2** — ovos grandes
- **2 xícaras** — gotas de chocolate meio amargo

## 👨‍🍳 Modo de preparo
1. Preaqueça o forno a 190 °C.
2. Em uma tigela pequena, misture a farinha, o bicarbonato e o sal.
3. Em uma tigela grande, bata a manteiga com o açúcar refinado e o açúcar mascavo até obter uma mistura leve e cremosa.
4. Acrescente a baunilha e os ovos, um de cada vez.
5. Adicione gradualmente os ingredientes secos e misture até incorporar.
6. Acrescente as gotas de chocolate e misture.
7. Distribua porções da massa em uma assadeira sem untar.
8. Asse por 9 a 11 minutos.


**Gerar valores de tipo enumerado**

Em alguns casos, talvez você queira que o modelo escolha uma única opção em uma lista. Para implementar esse comportamento, transmita um `enum` no seu esquema. É possível usar uma opção de enumeração em qualquer lugar em que um string possa ser usado no `responseSchema`, porque uma enumeração é uma matriz de strings. Assim como um esquema JSON, uma enumeração permite restringir a saída do modelo para atender aos requisitos do seu aplicativo.

Por exemplo, suponha que você esteja desenvolvendo um aplicativo para classificar instrumentos musicais em uma de cinco categorias: `Percussão`, `Cordas`, `Sopro`, `Metal` ou `Teclado`. Você pode criar uma enumeração para ajudar nessa tarefa.

No exemplo a seguir, você transmite uma enumeração como `response_schema`, restringindo o modelo a escolher a opção mais adequada.

In [ ]:
from google import genai
import enum

class Instrumento(enum.Enum):
  PERCUSSION = "Percussão"
  STRING = "Cordas"
  WOODWIND = "Sopro"
  BRASS = "Metal"
  KEYBOARD = "Teclado"

cliente = genai.Client( api_key= GEMINI_API_KEY )

resposta = cliente.models.generate_content(
    model = 'gemini-3.5-flash-lite',
    contents = 'Que tipo de instrumento é um oboé?',
    config = {
        'response_mime_type': 'text/x.enum',
        'response_schema': Instrumento,
    },
)

print(resposta.text)

Sopro


Também é possível transmitir o esquema como JSON:

In [ ]:
from google import genai

cliente = genai.Client( api_key= GEMINI_API_KEY )

resposta = cliente.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents='Que tipo de instrumento é um oboé?',
    config={
        'response_mime_type': 'text/x.enum',
        'response_schema': {
            "type": "STRING",
            "enum": ["Percussão", "Cordas", "Sopro", "Metal", "Teclado"],
        },
    },
)

print(resposta.text)

Sopro


### **Geração de Voz**

A **API Gemini** pode transformar entradas de texto em áudio de um ou vários locutores usando os recursos de geração de conversão de texto em voz (TTS) do Gemini.

**TTS de um único locutor**

In [ ]:
from google import genai

cliente = genai.Client( api_key= GEMINI_API_KEY )

resposta = cliente.models.generate_content(
    model = "gemini-3.8-flash-lite-tts",
    contents = "Tenha um dia maravilhoso!",
    config = {
        "response_modalities": ["AUDIO"],
        "speech_config": {
            "voice_config": {
                "prebuilt_voice_config": {
                    "voice_name": "Kore"
                }
            }
        },
    },
)

dados = resposta.candidates[0].content.parts[0].inline_data.data
with open("saida.wav", "wb") as f:
    f.write(dados)

In [ ]:
from IPython.display import Audio, display

display(Audio(data=dados))

In [ ]:
from IPython.display import Audio, display

display(Audio("saida.wav", autoplay=True))

### **Áudio**

O Gemini pode analisar e entender a entrada de áudio, permitindo casos de uso como estes:

   * Descrever, resumir ou responder a perguntas sobre conteúdo de áudio.
   * Fornecer uma transcrição do áudio.
   * Analisar segmentos específicos do áudio.


O Gemini oferece suporte aos seguintes tipos MIME de formato de áudio:

    WAV - audio/wav
    MP3 - audio/mp3
    AIFF - audio/aiff
    AAC - audio/aac
    OGG Vorbis - audio/ogg
    FLAC - audio/flac


In [ ]:
from google import genai

cliente = genai.Client( api_key= GEMINI_API_KEY )

audio = cliente.files.upload(file="/content/saida.wav")

resposta = cliente.models.generate_content(
    model = "gemini-3.5-flash-lite",
    contents = ["Descreva este clipe de áudio", audio]
)

print(resposta.text)

O áudio contém uma voz feminina dizendo "Tenha um dia maravilhoso".



Para receber uma transcrição dos dados de áudio, basta solicitar no prompt:

In [ ]:
from google import genai

cliente = genai.Client( api_key= GEMINI_API_KEY )

audio = cliente.files.upload(file="/content/saida.wav")

prompt = 'Gerar uma transcrição do discurso.'

resposta = cliente.models.generate_content(
  model = 'gemini-3.5-flash-lite',
  contents = [prompt, audio]
)

print(resposta.text)

Tenha um dia maravilhoso.
